# Explorador del Nomenclator — Registraduría Colombia 2026
**Autor: Nicolás Cardona - @cardonanl**

Herramienta para buscar cualquier término dentro del `nomenclator.json` y entender qué representa cada código encontrado.

Útil para:
- Encontrar el código de un candidato, municipio, corregimiento o puesto de votación
- Entender la jerarquía territorial de un código
- Validar que un código existe antes de usarlo en el extractor principal

---
## CELDA 1 — Parámetro de búsqueda
### ⚠️ Solo debes modificar esta celda

---
## CELDA 2 — Imports

In [ ]:
import requests
import pandas as pd
import json
from pathlib import Path

NOMENCLATOR_URL = "https://resultados.registraduria.gov.co/json/nomenclator.json"

# Niveles del nomenclator
NIVEL_NOMBRES = {
    1: 'País',
    2: 'Circunscripción',
    3: 'Municipio',
    4: 'Zona',
    5: 'Corregimiento',
    6: 'Puesto de votación',
}

print("Listo.")


Listo.


---
## CELDA 3 — Cargar nomenclator

In [19]:
def cargar_nomenclator(ruta_local=None, guardar_local=True):
    data = None

    if ruta_local and os.path.exists(ruta_local):
        print(f"Cargando desde archivo local: {ruta_local}")
        with open(ruta_local, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        print("Descargando nomenclator...")
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Referer": "https://resultados.registraduria.gov.co/",
            "Accept": "application/json, text/plain, */*",
        }
        response = requests.get(
            "https://resultados.registraduria.gov.co/json/nomenclator.json",
            headers=headers, timeout=60
        )
        response.raise_for_status()
        data = response.json()
        if guardar_local:
            ruta_guardar = ruta_local or "nomenclator 2026.json"
            with open(ruta_guardar, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)
            print(f"Guardado en: {ruta_guardar}")

    # ── Nueva estructura 2026 ──
    # data['amb'] es una lista: [{'elec': 1, 'ambitos': [...]}, {'elec': 2, ...}, ...]
    # Aplanamos todos los ámbitos en un dict indexado por 'i'
    ambitos = {}
    for grupo in data.get('amb', []):
        for a in grupo.get('ambitos', []):
            ambitos[str(a['i'])] = a

    print(f"Nomenclator cargado — {len(ambitos)} entradas totales")
    return data, ambitos


data, ambitos = cargar_nomenclator()
print(f"Total ámbitos: {len(ambitos):,}")


Descargando nomenclator...
Guardado en: nomenclator 2026.json
Nomenclator cargado — 1224 entradas totales
Total ámbitos: 1,224


---
## CELDA 4 — Motor de búsqueda
Busca en todos los campos del nomenclator: nombre, código, slug y cualquier otro campo de texto.

In [ ]:
def clasificar_nivel(nivel):
    """Devuelve la descripción del nivel territorial."""
    return NIVEL_NOMBRES.get(nivel, f'Nivel desconocido ({nivel})')


def inferir_tipo_codigo(codigo, nivel):
    c = str(codigo)
    if nivel == 1:
        return 'País'
    elif nivel == 2:
        return f'Circunscripción electoral (cód: {c})'
    elif nivel == 3:
        return f'Municipio (cód. DANE: {c[:2]}-{c[2:]})'
    elif nivel == 4:
        return f'Zona electoral del municipio {c[:5]}'
    elif nivel == 5:
        return f'Corregimiento / Inspección del municipio {c[:5]}'
    elif nivel == 6:
        return f'Puesto de votación — municipio {c[:5]}'
    return 'Tipo desconocido'


def buscar_en_nomenclator(ambitos, termino, case_sensitive=False):
    """
    Busca el término en todos los campos de texto del nomenclator.
    Devuelve un DataFrame con los resultados enriquecidos.
    """
    termino_busq = termino if case_sensitive else termino.lower()
    resultados   = []
    vistos       = set()

    for ambito_key, item in ambitos.items():          # ← dict aplanado, un item por clave
        codigo = str(item.get('co', item.get('c', '')))  # 'co' en 2026, 'c' fallback
        nombre = str(item.get('n', ''))
        slug   = str(item.get('s', ''))
        nivel  = item.get('l', 0)

        campos_texto = {
            'nombre': nombre,
            'slug':   slug,
            'codigo': codigo,
        }

        for campo, valor in campos_texto.items():
            valor_cmp = valor if case_sensitive else valor.lower()
            if termino_busq in valor_cmp:
                clave_dedup = (codigo, campo)
                if clave_dedup in vistos:
                    continue
                vistos.add(clave_dedup)

                resultados.append({
                    'match_en':         campo,
                    'codigo':           codigo,
                    'nombre':           nombre,
                    'slug':             slug,
                    'nivel':            nivel,
                    'tipo_territorial': clasificar_nivel(nivel),
                    'tipo_codigo':      inferir_tipo_codigo(codigo, nivel),
                    'cod_depto':        codigo[:2] if len(codigo) >= 2 else '',
                    'cod_municipio':    codigo[:5] if len(codigo) >= 5 else '',
                    'ambito':           ambito_key,
                })
                break  # un match por item es suficiente

    df = pd.DataFrame(resultados)
    if not df.empty:
        df = df.sort_values(['nivel', 'codigo']).reset_index(drop=True)
    return df


print("Motor de búsqueda listo.")


Motor de búsqueda listo.


In [32]:
# Término a buscar (puede ser nombre parcial, código, municipio, candidato, etc.)
# Ejemplos: 'duvalier', 'cali', '76001', 'barranquilla', 'zona99'
TERMINO_BUSQUEDA = 'cali'   # <--- CAMBIAR AQUÍ

# Sensible a mayúsculas/minúsculas?
# False = busca sin importar capitalización (recomendado)
CASE_SENSITIVE = False


In [33]:
df_resultados = buscar_en_nomenclator(ambitos, TERMINO_BUSQUEDA, CASE_SENSITIVE)

---
## CELDA 5 — Ejecutar búsqueda y mostrar resultados

In [34]:
if df_resultados.empty:
    print(f"❌ No se encontraron resultados para '{TERMINO_BUSQUEDA}'")
else:
    print(f"✅ {len(df_resultados)} resultado(s) para '{TERMINO_BUSQUEDA}':\n")
    
    # Mostrar agrupado por tipo territorial
    for nivel, grupo in df_resultados.groupby('nivel'):
        tipo = NIVEL_NOMBRES.get(nivel, f'Nivel {nivel}')
        print(f"── {tipo.upper()} ({len(grupo)} resultado(s)) ──")
        for _, row in grupo.iterrows():
            print(f"   Código     : {row['codigo']}")
            print(f"   Nombre     : {row['nombre']}")
            print(f"   Tipo       : {row['tipo_codigo']}")
            print(f"   Match en   : campo '{row['match_en']}'")
            if row['nivel'] >= 3:
                print(f"   Depto      : {row['cod_depto']}  |  Municipio: {row['cod_municipio']}")
            print()

df_resultados


✅ 5 resultado(s) para 'cali':

── MUNICIPIO (5 resultado(s)) ──
   Código     : 2500076
   Nombre     : SAN CALIXTO
   Tipo       : Municipio (cód. DANE: 25-00076)
   Match en   : campo 'nombre'
   Depto      : 25  |  Municipio: 25000

   Código     : 2504076
   Nombre     : SAN CALIXTO
   Tipo       : Municipio (cód. DANE: 25-04076)
   Match en   : campo 'nombre'
   Depto      : 25  |  Municipio: 25040

   Código     : 2700031
   Nombre     : CALIFORNIA
   Tipo       : Municipio (cód. DANE: 27-00031)
   Match en   : campo 'nombre'
   Depto      : 27  |  Municipio: 27000

   Código     : 3100001
   Nombre     : CALI
   Tipo       : Municipio (cód. DANE: 31-00001)
   Match en   : campo 'nombre'
   Depto      : 31  |  Municipio: 31000

   Código     : 3100040
   Nombre     : CALIMA (DARIEN)
   Tipo       : Municipio (cód. DANE: 31-00040)
   Match en   : campo 'nombre'
   Depto      : 31  |  Municipio: 31000



,match_en,codigo,nombre,slug,nivel,tipo_territorial,tipo_codigo,cod_depto,cod_municipio,ambito
0,nombre,2500076,SAN CALIXTO,SAN-CALIXTO,3,Municipio,Municipio (cód. DANE: 25-00076),25,25000,917
1,nombre,2504076,SAN CALIXTO,SAN-CALIXTO,3,Municipio,Municipio (cód. DANE: 25-04076),25,25040,138
2,nombre,2700031,CALIFORNIA,CALIFORNIA,3,Municipio,Municipio (cód. DANE: 27-00031),27,27000,201
3,nombre,3100001,CALI,CALI,3,Municipio,Municipio (cód. DANE: 31-00001),31,31000,200
4,nombre,3100040,CALIMA (DARIEN),CALIMA-DARIEN,3,Municipio,Municipio (cód. DANE: 31-00040),31,31000,202


In [29]:
# Ver qué niveles hay realmente en los resultados
print(df_resultados[['nivel', 'nombre', 'codigo']].to_string())

# Y también ver si el depto Valle existe en ambitos
for k, v in ambitos.items():
    if 'valle' in v.get('n', '').lower():
        print(v)

   nivel                          nombre   codigo
0      3                      VALLEDUPAR  1200001
1      3                      VALLEDUPAR  1212001
2      3               VALLE DE SAN JOSE  2700217
3      3                    RONCESVALLES  2900100
4      3               VALLE DE SAN JUAN  2900118
5      3  VALLE DEL GUAMUEZ (LA HORMIGA)  6400028
6      3  VALLE DEL GUAMUEZ (LA HORMIGA)  6411028
{'i': 176, 'n': 'VALLE DEL GUAMUEZ (LA HORMIGA)', 'co': '6411028', 's': 'VALLE-DEL-GUAMUEZ-LA-HORMIGA', 'l': 3, 'p': [{'l': 2, 'p': [3]}], 'r': [-1, -1, -1, 176], 'h': []}
{'i': 177, 'n': 'VALLEDUPAR', 'co': '1212001', 's': 'VALLEDUPAR', 'l': 3, 'p': [{'l': 2, 'p': [4]}], 'r': [-1, -1, -1, 177], 'h': []}
{'i': 876, 'n': 'RONCESVALLES', 'co': '2900100', 's': 'RONCESVALLES', 'l': 3, 'p': [{'l': 2, 'p': [31]}], 'r': [876, 876, 876, -1], 'h': []}
{'i': 1163, 'n': 'VALLE DE SAN JOSE', 'co': '2700217', 's': 'VALLE-DE-SAN-JOSE', 'l': 3, 'p': [{'l': 2, 'p': [29]}], 'r': [1163, 1163, 1163, -1], 'h': []

In [36]:
for k, v in ambitos.items():
    if v.get('l') == 2:
        print(v.get('co'), v.get('n'))

0001 CIRCUNSCRIPCIÓN 1 (CAUCA- NARI
0010 CIRCUNSCRIPCIÓN 10 NARIÑO
0011 CIRCUNSCRIPCIÓN 11 PUTUMAYO
0012 CIRCUNSCRIPCIÓN 12 (CESAR - LA
0013 CIRCUNSCRIPCIÓN 13 (BOLÍVAR -
0014 CIRCUNSCRIPCIÓN 14 CÓRDOBA
0015 CIRCUNSCRIPCIÓN 15 TOLIMA
0016 CIRCUNSCRIPCIÓN 16 ANTIOQUIA
0002 CIRCUNSCRIPCIÓN 2 ARAUCA
0003 CIRCUNSCRIPCIÓN 3 ANTIOQUIA
0004 CIRCUNSCRIPCIÓN 4 NORTE DE SAN
0005 CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
0006 CIRCUNSCRIPCIÓN 6 (CHOCÓ - ANT
0007 CIRCUNSCRIPCIÓN 7 (META - GUAV
0008 CIRCUNSCRIPCIÓN 8 (BOLÍVAR - S
0009 CIRCUNSCRIPCIÓN 9 (CAUCA - VAL


---
## CELDA 6 — Ver contexto jerárquico de un código
Dado uno de los códigos encontrados, muestra todos sus ancestros y descendientes directos en la jerarquía territorial.

In [28]:
def contexto_jerarquico(data, codigo_objetivo):
    """
    Para un código dado, encuentra sus ancestros (departamento, municipio, zona)
    y sus hijos directos (puestos de votación dentro de él).
    """
    codigo_objetivo = str(codigo_objetivo)
    todos = {}
    for ambito_key, items in data.get('ambitos', {}).items():
        for item in items:
            c = str(item.get('c', ''))
            todos[c] = item

    item_obj = todos.get(codigo_objetivo)
    if not item_obj:
        print(f"Código '{codigo_objetivo}' no encontrado en el nomenclator.")
        return

    nivel_obj = item_obj.get('l', 0)
    nombre_obj = item_obj.get('n', '')
    print(f"{'='*60}")
    print(f"Contexto jerárquico para: {nombre_obj} ({codigo_objetivo})")
    print(f"Tipo: {inferir_tipo_codigo(codigo_objetivo, nivel_obj)}")
    print(f"{'='*60}")

    # Ancestros: buscar por prefijo del código
    print("\n📍 ANCESTROS:")
    for nivel in range(1, nivel_obj):
        for c, item in todos.items():
            if item.get('l') == nivel and codigo_objetivo.startswith(c):
                print(f"   Nivel {nivel} ({NIVEL_NOMBRES.get(nivel,'?')}): {item.get('n','')} → código {c}")

    # Hijos directos: items cuyo padre contiene este código
    print(f"\n👇 HIJOS DIRECTOS (nivel {nivel_obj + 1}):")
    hijos = [
        item for c, item in todos.items()
        if item.get('l') == nivel_obj + 1 and str(c).startswith(codigo_objetivo)
    ]
    if hijos:
        for h in sorted(hijos, key=lambda x: str(x.get('c', ''))):
            print(f"   {h.get('n','')} → código {h.get('c','')} | mesas: {h.get('m', '?')}")
    else:
        print("   (sin hijos en el siguiente nivel)")

    print(f"\nTotal hijos directos encontrados: {len(hijos)}")


# Pega aquí un código de los resultados de la búsqueda para ver su contexto
# Ejemplo:
if not df_resultados.empty:
    primer_codigo = df_resultados.iloc[0]['codigo']
    print(f"Mostrando contexto del primer resultado: {primer_codigo}\n")
    contexto_jerarquico(nomenclator, primer_codigo)
else:
    print("Primero ejecuta la búsqueda en la CELDA 5.")

# Para inspeccionar otro código, descomentar:
# contexto_jerarquico(nomenclator, '76001')


Mostrando contexto del primer resultado: 1200001



AttributeError: 'tuple' object has no attribute 'get'

---
## CELDA 7 — (Opcional) Exportar resultados de búsqueda

In [ ]:
if not df_resultados.empty:
    nombre_archivo = f"busqueda_{TERMINO_BUSQUEDA.replace(' ', '_')}_nomenclator.csv"
    df_resultados.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    print(f"Exportado: {nombre_archivo}")
else:
    print("No hay resultados para exportar.")
